# Multimodal Search — Query Examples

Interactive queries against the **`sharepoint-page-index`** built by this kit. The index is *multimodal*: every page produces **text rows** (`kind=text`, 3072-dim text vectors) and extracted **image rows** (`kind=image`, 1024-dim Vision vectors), all security-trimmed by SharePoint ACLs.

The examples below show:

1. **Hybrid text search** — BM25 + text vectors + semantic reranking
2. **Text → image (cross-modal)** — a *text* query retrieves matching *images* via the Vision vectorizer
3. **Semantic Q&A** — natural-language question over the text
4. **Filtered search** — scope by `kind` / page range with an OData filter
5. **Combined multimodal** — one query returns both text and image hits

> Auth uses `DefaultAzureCredential` (run `az login` first). Your token is sent as `x-ms-query-source-authorization`, so results are trimmed to what **you** are allowed to see in SharePoint.

## Setup

Install dependencies (once), then load config from `.env` / `.env.derived` and define the query helpers.

In [ ]:
%pip install -q azure-identity requests

In [ ]:
import os, base64, textwrap
from pathlib import Path

import requests
from azure.identity import DefaultAzureCredential
from IPython.display import display, Markdown, Image


def _find_root(start: Path) -> Path:
    """Walk up from the notebook's directory to find the folder holding .env."""
    for p in [start, *start.parents]:
        if (p / ".env").exists():
            return p
    return start


def _load_env(path: Path) -> None:
    if not path.exists():
        return
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, val = line.split("=", 1)
        key, val = key.strip(), val.lstrip()
        if val[:1] in ('"', "'"):
            q = val[0]
            end = val.find(q, 1)
            val = val[1:end] if end >= 1 else val[1:]
        else:
            h = val.find("#")
            val = (val[:h] if h >= 0 else val).strip()
        os.environ[key] = val


ROOT = _find_root(Path.cwd())
_load_env(ROOT / ".env")            # base config
_load_env(ROOT / ".env.derived")    # provisioning outputs (SEARCH_ENDPOINT, ...)

SEARCH_ENDPOINT = os.environ["SEARCH_ENDPOINT"].rstrip("/")
INDEX_NAME = os.environ["INDEX_NAME"]
API_VERSION = os.environ.get("SEARCH_API_VERSION", "2026-05-01-preview")

_cred = DefaultAzureCredential()


def _token() -> str:
    return _cred.get_token("https://search.azure.com/.default").token


print("Endpoint:", SEARCH_ENDPOINT)
print("Index   :", INDEX_NAME)

In [ ]:
def search(query, top=5, filter=None, text_vector=True, image_vector=False,
           semantic=True, select="id,kind,sourceFile,page,pageTo,webUrl,content"):
    """Run a query against the index.

    text_vector  -> add a vector query over contentVector (OpenAI text embeddings)
    image_vector -> add a vector query over imageVector  (Vision; text is embedded
                    into image space, enabling cross-modal text->image search)
    """
    token = _token()
    url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version={API_VERSION}"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}",
        # Same user token -> per-document SharePoint ACL trimming.
        "x-ms-query-source-authorization": token,
    }
    payload = {"search": query, "top": top, "select": select}
    if filter:
        payload["filter"] = filter
    vq = []
    if text_vector:
        vq.append({"kind": "text", "text": query, "fields": "contentVector", "k": top})
    if image_vector:
        vq.append({"kind": "text", "text": query, "fields": "imageVector", "k": top})
    if vq:
        payload["vectorQueries"] = vq
    if semantic:
        payload["queryType"] = "semantic"
        payload["semanticConfiguration"] = "default"
    r = requests.post(url, headers=headers, json=payload, timeout=60)
    r.raise_for_status()
    return r.json().get("value", [])


def show_text(hits):
    if not hits:
        display(Markdown("_No results — check you're `az login`'d and have **Search Index Data Reader**._"))
        return
    for i, h in enumerate(hits, 1):
        rr, sc = h.get("@search.rerankerScore"), h.get("@search.score")
        score = f"reranker={rr:.3f}" if rr is not None else f"score={sc:.3f}"
        body = textwrap.shorten(" ".join((h.get("content") or "").split()), width=400, placeholder=" \u2026")
        md = f"**[{i}] {h.get('sourceFile','(unknown)')}** \u2014 page {h.get('page')} \u00b7 {score}\n\n{body}"
        if h.get("webUrl"):
            md += f"\n\n[open in SharePoint]({h['webUrl']})"
        display(Markdown(md))


def show_images(hits):
    if not hits:
        display(Markdown("_No image results._"))
        return
    for i, h in enumerate(hits, 1):
        sc = h.get("@search.score")
        display(Markdown(f"**[{i}] {h.get('sourceFile','(unknown)')}** \u2014 page {h.get('page')} \u00b7 score={sc:.3f} \u00b7 `{h.get('imagePath','')}`"))
        data = h.get("imageData")
        if data:
            try:
                display(Image(data=base64.b64decode(data), width=360))
            except Exception as e:  # noqa: BLE001
                display(Markdown(f"_(couldn't render image: {e})_"))

## 1. Hybrid text search

BM25 keyword matching **fused with** text-vector similarity, then re-ordered by the semantic ranker. The `kind eq 'text'` filter keeps results to text chunks.

In [ ]:
hits = search("What is osteoporosis and why is it a health care burden?",
              top=5, filter="kind eq 'text'")
show_text(hits)

## 2. Text \u2192 image (cross-modal) search

A **text** query is embedded into the **image** vector space by the Vision vectorizer, so it retrieves matching figures/diagrams directly. We filter to `kind eq 'image'` and render the extracted images inline from the base64 stored in the index.

In [ ]:
hits = search("illustration of the human skeleton and bones",
              top=4,
              filter="kind eq 'image'",
              text_vector=False, image_vector=True,
              semantic=False,
              select="id,kind,sourceFile,page,imagePath,imageData")
show_images(hits)

## 3. Semantic Q&A over the text

A natural-language question. Hybrid retrieval finds candidates and the semantic ranker surfaces the most answer-like passages (`@search.rerankerScore`).

In [ ]:
hits = search("Which therapies or drug classes are used to treat bone diseases?",
              top=5, filter="kind eq 'text'")
show_text(hits)

## 4. Filtered search (OData `$filter`)

Combine vector/semantic relevance with structured filters — here, text rows within a page range. Filterable fields include `kind`, `page`, `pageTo`, `sourceFile`, and `lastModified`.

In [ ]:
hits = search("fracture risk and bone mineral density",
              top=5,
              filter="kind eq 'text' and page ge 85 and page le 100")
show_text(hits)

## 5. Combined multimodal query

One request with **two** vector queries (text *and* image). Without a `kind` filter, results interleave text passages and figures — the full multimodal retrieval. We split them for display.

In [ ]:
hits = search("skeletal system, bone structure and fractures",
              top=8,
              text_vector=True, image_vector=True,
              semantic=False,
              select="id,kind,sourceFile,page,webUrl,content,imagePath,imageData")

text_hits = [h for h in hits if h.get("kind") == "text"]
image_hits = [h for h in hits if h.get("kind") == "image"]

display(Markdown(f"### Text hits ({len(text_hits)})"))
show_text(text_hits)
display(Markdown(f"### Image hits ({len(image_hits)})"))
show_images(image_hits)

---
### Notes

- **Security trimming:** results are limited to documents whose `UserIds`/`GroupIds` match your identity. Querying with an admin key or from the portal (no user token) returns **0** rows by design.
- **Edit the queries:** swap in your own phrasing above. Toggle `text_vector` / `image_vector`, adjust `filter`, or set `semantic=False` to see raw hybrid ranking.
- **Fields:** `contentVector` (3072) and `imageVector` (1024) are searchable but not retrievable; retrievable fields include `content`, `imagePath`, `imageData`, `page`, `sourceFile`, `webUrl`.